# 👻 Horror & Dark Entity Pack MVP — Qwen3-TTS

> **⚠️ Content Warning:** These voices are designed for mature horror games and may be unsettling.

Generate atmospheric, threatening, and whispered lines for dark entities using Qwen3-TTS VoiceDesign.

| Entity | Description |
|---|---|
| Vengeful Ghost | Mournful, distant, breaking into hiss |
| Demon Lord | Deep, resonant, ancient, calm |
| Cult Leader | Charismatic, unsettling, unhinged edge |
| Possessed Child | Innocent slipping into guttural wrongness |
| Undead Knight | Hollow, slow, dead-air voice |
| Eldritch Horror | Alien, multiple harmonics, vast |

In [ ]:
!pip install -q qwen-tts soundfile

import torch
import gc
import os
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

In [ ]:
OUTPUT_DIR = "/content/horror_voice_pack"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def clear_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

HORROR_ROSTER = {
    "Vengeful Ghost": {
        "voice_prompt": "A whispering, ethereal female voice — barely audible, distant echo quality, mournful, sometimes breaking into an inhuman hiss",
        "atmospheric_line": "I have waited... so long... for someone to hear me.",
        "threat_line": "You cannot leave. No one leaves. Not anymore.",
        "whisper_line": "Why didn't you save me?"
    },
    "Demon Lord": {
        "voice_prompt": "An impossibly deep, resonant, ancient voice — each word deliberate, a slight reverb as if speaking from a vast cavern, utterly calm",
        "atmospheric_line": "Millennia I have slept beneath your world. Your fear is... exquisite.",
        "threat_line": "I will not kill you quickly. That would be merciful. I am not merciful.",
        "whisper_line": "Your soul already belongs to me."
    },
    "Cult Leader": {
        "voice_prompt": "A charismatic, unsettling, too-smooth voice — warm on the surface, with a dangerous undercurrent, slight unhinged edge",
        "atmospheric_line": "We are all children of the Void. You simply haven't accepted it yet.",
        "threat_line": "The congregation has spoken. You've been chosen for a very special purpose.",
        "whisper_line": "Join us. You'll feel so much better once you stop resisting."
    },
    "Possessed Child": {
        "voice_prompt": "A child's innocent voice that keeps slipping — starts sweet and soft, then suddenly drops to something guttural and wrong, then back to sweet",
        "atmospheric_line": "I made a new friend. He lives in the dark. He says you taste like fear.",
        "threat_line": "WE WILL BREAK EVERY BONE IN YOUR BODY. — I mean... will you play with me?",
        "whisper_line": "He's standing right behind you."
    },
    "Undead Knight": {
        "voice_prompt": "A hollow, slow, dead-air voice — no emotion, long pauses between words, as if speaking is an effort across a great void",
        "atmospheric_line": "I... remember... what it felt like... to be warm.",
        "threat_line": "I have killed... seven hundred men. You will be... seven hundred and one.",
        "whisper_line": "I feel... nothing. That is the worst of it."
    },
    "Eldritch Horror": {
        "voice_prompt": "A deeply alien voice — multiple harmonic layers, words formed with wrong emphasis, pacing that doesn't match human speech patterns, vast and incomprehensible",
        "atmospheric_line": "You call this existence. We have observed seventeen thousand civilizations call it that. All were wrong.",
        "threat_line": "Your mind is not large enough to contain what I am. That will not stop me from trying to fit inside it.",
        "whisper_line": "We have always been here."
    }
}

In [ ]:
clear_vram()

model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
print(f"Loading model: {model_id}")

model = Qwen3TTSModel.from_pretrained(
    model_id, 
    device_map="cuda:0", 
    dtype=torch.bfloat16, 
    attn_implementation="sdpa"
)

print("Model loaded successfully!")

In [ ]:
def generate_and_save(text, instruct, filename):
    res = model.generate_voice_design(text, "English", instruct)
    
    if isinstance(res, tuple):
        audio, sr = res
    else:
        audio, sr = res, 24000
        
    if hasattr(audio, 'cpu'):
        audio = audio.cpu().numpy()
        
    audio = np.squeeze(audio)
    filepath = os.path.join(OUTPUT_DIR, filename)
    sf.write(filepath, audio, sr)
    return filepath, audio, sr

atmospheric_audios = []
global_sr = 24000

for entity, data in HORROR_ROSTER.items():
    print("="*50)
    print(f"--- 👻 Entity: {entity} ---")
    print(f"Base Prompt: {data['voice_prompt']}\n")
    
    base_name = entity.lower().replace(" ", "_")
    
    # Atmospheric Line
    print("🌫️ Atmospheric:", data['atmospheric_line'])
    path_a, aud_a, sr = generate_and_save(data['atmospheric_line'], data['voice_prompt'], f"horror_{base_name}_atmospheric.wav")
    atmospheric_audios.append(aud_a)
    global_sr = sr
    display(Audio(path_a))
    
    # Threat Line
    print("🩸 Threat:", data['threat_line'])
    path_t, _, _ = generate_and_save(data['threat_line'], data['voice_prompt'], f"horror_{base_name}_threat.wav")
    display(Audio(path_t))
    
    # Whisper Line
    whisper_prompt = data['voice_prompt'] + ", speaking in a barely audible whisper"
    print("🤫 Whisper Prompt:", whisper_prompt)
    print("Line:", data['whisper_line'])
    path_w, _, _ = generate_and_save(data['whisper_line'], whisper_prompt, f"horror_{base_name}_whisper.wav")
    display(Audio(path_w))
    
    print("\n")

In [ ]:
print("🎬 Generating Atmospheric Haunted Scene...")
silence = np.zeros(int(1.5 * global_sr))
scene_parts = []

for audio in atmospheric_audios:
    scene_parts.append(audio)
    scene_parts.append(silence)
    
full_scene_audio = np.concatenate(scene_parts)
scene_path = os.path.join(OUTPUT_DIR, "horror_full_atmospheric_scene.wav")
sf.write(scene_path, full_scene_audio, global_sr)

print("Full Atmospheric Scene (1.5s silence between entities):")
display(Audio(scene_path))

*Note: The above audio simulates a haunted location where the player hears different entities as they move through rooms.*

In [ ]:
import shutil
from google.colab import files

print("📦 Zipping and downloading the horror voice pack...")
shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
files.download(f"{OUTPUT_DIR}.zip")